<a href="https://colab.research.google.com/github/Doladun1/PyTorch/blob/main/simple_NueralNetwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [245]:
import torch
import torch.nn as nn
import torch.nn.functional as f

In [246]:
my_torch = torch.rand(1,4)
my_torch

tensor([[0.8697, 0.3995, 0.7508, 0.0322]])

In [247]:
# Create a Class Model
class Model(nn.Module):
  #Add layers
  #Input Layer (Flower features) -> 4 Nueral Layers -> Output Layer (which flower I got)
  def __init__(self, in_features=4, h1=8, h2=9, out_features=3):
    super().__init__()
    self.fc1 = nn.Linear(in_features, h1)
    self.fc2 = nn.Linear(h1,h2)
    self.out = nn.Linear(h2,out_features)

  def forward(self,x):
    x = f.relu(self.fc1(x))
    x = f.relu(self.fc2(x))
    x = self.out(x)

    return x


In [248]:
#Pick a seed
torch.manual_seed(87)
#create Model
model = Model()

In [249]:
#Import data to test w/ model
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [250]:
#Load the data
url = 'https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv'
my_df = pd.read_csv(url)
my_df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [251]:
my_df = my_df.replace('setosa',0)
my_df = my_df.replace('versicolor',1)
my_df = my_df.replace('virginica',2)
my_df

/tmp/ipykernel_11226/170219684.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  my_df = my_df.replace('virginica',2)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [252]:
#Train Test and Split
X = my_df.drop('petal_width',axis=1)
y = my_df['petal_width']

In [253]:
#Convert to numpy array
X = X.values
y = y.values

In [254]:
from sklearn.model_selection import train_test_split


In [255]:
#Train Test SPlit
X_train, X_test, y_train, y_test = train_test_split( X, y ,test_size=0.2, random_state=42)

In [256]:
#Convert X features to Float Tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)


In [257]:
#Convert y labels to Tensors
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)


In [258]:
#Set Criterion for the model, predict how much error etc
criterion = nn.CrossEntropyLoss()
#Choose Optimizer lr = learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
#

In [259]:
#Train the MODEL
epochs = 80
losses = []

for i in range(epochs):
  y_pred = model.forward(X_train)
  # Get predicted results

  #Measure Loss and Error
  loss = criterion(y_pred, y_train)
  losses.append(loss.detach().numpy())

  #print 10 epochs
  if i % 10 == 0:
    print(f'Epoch: {i} Loss: {loss}')

#Back propogation: Take error moving back and feed it back to fine tune errthing
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()



Epoch: 0 Loss: 1.049864649772644
Epoch: 10 Loss: 0.2652023434638977
Epoch: 20 Loss: 0.22804975509643555
Epoch: 30 Loss: 0.22501787543296814
Epoch: 40 Loss: 0.22491218149662018
Epoch: 50 Loss: 0.21651306748390198
Epoch: 60 Loss: 0.21039338409900665
Epoch: 70 Loss: 0.20484715700149536


In [260]:
#Evaluate TEst on Data Set
with torch.no_grad(): # Turn off back propogation
  y_eval = model.forward(X_test)
  loss = criterion(y_eval, y_test)

print(f'Loss: {loss}')

Loss: 0.23693689703941345


In [261]:
#Try again and print
correct =0
with torch.no_grad(): # Turn off back propogation
  for i, data in enumerate(X_test):
    y_eval = model.forward(data)

    if y_eval.argmax().item() == y_test[i]:
      correct +=1

    print(f'{i+1}. {str(data)} -> {str(y_eval)} {y_eval.argmax().item()} {y_test[i]}')

1. tensor([6.1000, 2.8000, 4.7000, 1.0000]) -> tensor([-10.4298,   5.8789,  -2.0913]) 1 1
2. tensor([5.7000, 3.8000, 1.7000, 0.0000]) -> tensor([ 15.7850,   3.8102, -36.6056]) 0 0
3. tensor([7.7000, 2.6000, 6.9000, 2.0000]) -> tensor([-29.8259,  12.2126,  11.9926]) 1 2
4. tensor([6.0000, 2.9000, 4.5000, 1.0000]) -> tensor([-8.7238,  5.4530, -3.7898]) 1 1
5. tensor([6.8000, 2.8000, 4.8000, 1.0000]) -> tensor([-9.2690,  6.0687, -4.3211]) 1 1
6. tensor([5.4000, 3.4000, 1.5000, 0.0000]) -> tensor([ 15.4393,   3.5482, -35.0498]) 0 0
7. tensor([5.6000, 2.9000, 3.6000, 1.0000]) -> tensor([-3.6662,  4.6731, -8.9776]) 1 1
8. tensor([6.9000, 3.1000, 5.1000, 2.0000]) -> tensor([-23.2314,   9.2568,   9.1278]) 1 2
9. tensor([6.2000, 2.2000, 4.5000, 1.0000]) -> tensor([-11.7907,   6.5334,  -0.4339]) 1 1
10. tensor([5.8000, 2.7000, 3.9000, 1.0000]) -> tensor([-5.9021,  5.0501, -6.4847]) 1 1
11. tensor([6.5000, 3.2000, 5.1000, 2.0000]) -> tensor([-23.4752,   9.1476,   9.5012]) 2 2
12. tensor([4.8000, 

In [262]:
#Correct or not

print(correct)

25


In [263]:
y_test

tensor([1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 1, 0, 2,
        2, 2, 1, 2, 0, 0])

In [264]:
y_eval.argmax().item()

0